# AttackAware PolyIoM v1.1.4 — score histograms

The paper reports every recognition result as a single point estimate, and has
no figure of how the system actually scores, because the score distributions
were never kept. This notebook keeps them, for **two** systems.

## Why two systems

The first criterion a cancelable-biometric paper has to address is
**performance preservation**, and in this literature it is always reported as
the protected system measured against the **unprotected** one. The study so far
compares three *protected* variants against each other; nobody has measured
plain cosine matching on these splits. Without that reference the headline
figure of the results section cannot be drawn.

| System | Score | Role |
|---|---|---|
| protected | integer collision count in `0..M` | the sealed pipeline |
| unprotected | cosine similarity between the same embeddings | the performance reference |

## Why histograms, not raw score vectors

A collision score is an integer in `0..M`, so a count per score value is a
**lossless** record. Cosine is continuous, so it is binned over `[-1, 1]` at
2000 bins — 0.001 resolution, far finer than any DET curve needs. A few
kilobytes instead of megabytes, and nothing approximated.

Everything the results figures need follows exactly:

| Figure | Criterion | Derived from |
|---|---|---|
| DET curves, protected against unprotected | performance preservation | genuine + impostor, both systems |
| Genuine and impostor distributions with `tau*` | separability | the same |
| Mated and non-mated distributions with `D_link(s)` overlaid | unlinkability | mated + non-mated + the local curve |
| Genuine, impostor and pseudo-impostor on one axis | revocability | the same histograms |

**One caution carried into the paper.** The mated distribution is both the
unlinkability mated set and the revocability pseudo-impostor set. It is one
measurement, and the paper must not present the two properties as independent
evidence.

## Scope and checks

Development and evaluation partitions for both modalities, plus external VCTK
for voice when its embeddings are present. The local unlinkability curve
`D_link(s)` is stored alongside the global value the evaluation already
reports.

The EER recomputed from the evaluation histogram is printed against the sealed
held-out figure. If they disagree, the dump is not faithful and no figure
should be built on it.

Writes one file, `runs/scores/score_histograms.json`, and no seal.


In [ ]:
#@title 1. Mount Drive and verify the score-dump inputs
from google.colab import drive
drive.mount("/content/drive")

import hashlib, json, math, os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display

PROJECT = Path("/content/drive/MyDrive/AttackAware_PolyIoM_v1_1_4")
DIR = {
    "protocol": PROJECT / "protocol",
    "embeddings": PROJECT / "embeddings",
    "runs": PROJECT / "runs",
    "seal": PROJECT / "seal",
}
S0, G = 2026, 5
DEVICE = torch.device("cpu")

LFW_FACE_TRIALS = DIR["protocol"] / "lfw_face_trials.tsv"
LFW_EMB = DIR["embeddings"] / "lfw_all_valid_embeddings.npz"
LIBRI_MANIFEST = DIR["protocol"] / "librispeech_internal.tsv"
LIBRI_EMB = DIR["embeddings"] / "librispeech_internal_embeddings.npz"

HELDOUT_RESULTS = {
    modality: DIR["runs"] / "heldout" / modality / "heldout_result.json"
    for modality in ("face", "voice")
}
ABLATION_OUTPUT = DIR["runs"] / "ablation" / "ablation_result.json"

REQUIRED = [
    LFW_FACE_TRIALS, LFW_EMB, LIBRI_MANIFEST, LIBRI_EMB,
    DIR["runs"] / "key_search/face/selected_key.json",
    DIR["runs"] / "key_search/voice/selected_key.json",
    HELDOUT_RESULTS["face"], HELDOUT_RESULTS["voice"],
]
missing = [str(p.relative_to(PROJECT)) for p in REQUIRED if not p.is_file()]
if missing:
    raise FileNotFoundError(
        "Score dump cannot start; required frozen artefacts are missing:\n- "
        + "\n- ".join(missing)
    )

torch.use_deterministic_algorithms(True, warn_only=False)
if hasattr(torch.backends, "cudnn"):
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

TENSOR_DIR = DIR["runs"] / "iom_tensors"
if not TENSOR_DIR.is_dir():
    raise FileNotFoundError(
        f"Frozen IoM tensors are missing: {TENSOR_DIR}. The proposed arm "
        f"cannot reproduce the sealed pipeline without them."
    )
available = sorted(p.name for p in TENSOR_DIR.glob("*.pt"))
print("Preflight: PASS")
print(f"Frozen IoM tensors on disk ({len(available)}):",
      ", ".join(available) if available else "NONE - the run will fail")
print("Hash reconstruction device:", DEVICE)
print("This notebook writes exactly one file: score_histograms.json")
print("It changes no seal and fits no threshold.")

In [ ]:
#@title 2. Load the pipeline primitives and the score dump
for name in ("ablation_only.py", "scoredump_only.py"):
    path = PROJECT / name
    exec(compile(path.read_text(), str(path), "exec"), globals())
print("Score dump runtime: READY")

In [ ]:
#@title 3. Dump the score histograms
scores = run_score_dump()